# Independent SHIELDOSE radiation reference

This notebook reproduces the historical/reference SAPPHIRE–SHIELDOSE workflow. The current mission simulator uses the separate frozen OLTARIS BFO environment. See README.md for the authoritative source map and strategy workflow. Existing code cells are preserved; run them only when intentionally regenerating reference outputs.


In [ ]:
from pathlib import Path
import os, sys, subprocess, importlib.util

for module, package in {"pandas": "pandas", "numpy": "numpy", "scipy": "scipy", "matplotlib": "matplotlib", "pypdf": "pypdf", "bs4": "beautifulsoup4"}.items():
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])
import pandas as pd
from IPython.display import display

# Optional override, for example in Colab:
# os.environ["CNL_PROJECT_ROOT"] = "/content/drive/MyDrive/cnl"
def find_root():
    if os.environ.get("CNL_PROJECT_ROOT"):
        candidates = [Path(os.environ["CNL_PROJECT_ROOT"]).expanduser()]
    else:
        candidates = [Path.cwd(), *Path.cwd().parents]
        candidates = [p for p in candidates if (p / "data/processed/spe_scenarios.csv").is_file()]
        if not candidates and Path("/content").exists():
            from google.colab import drive
            drive.mount("/content/drive")
            candidates = [p.parents[2] for p in Path("/content/drive/MyDrive").rglob("spe_scenarios.csv")
                          if p.parent.name == "processed" and p.parent.parent.name == "data"]
    candidates = list(dict.fromkeys(p.resolve() for p in candidates))
    if len(candidates) != 1 or not (candidates[0] / "data/processed/spe_scenarios.csv").is_file():
        raise FileNotFoundError(f"Set CNL_PROJECT_ROOT to your cnl folder. Candidates: {candidates}")
    return candidates[0]

ROOT = find_root()
spe_hourly = pd.read_csv(ROOT / "data/processed/spe_hourly.csv", parse_dates=["timestamp_utc"])
shielding_dose = pd.read_csv(ROOT / "data/processed/shielding_dose.csv")
spe_scenarios = pd.read_csv(ROOT / "data/processed/spe_scenarios.csv")
print("Project:", ROOT)


## What is available?
OMNIWeb provides **measured hourly flux**. OLTARIS and SHIELDOSE provide **simulated dose quantities**. SAPPHIRE provides **simulated event-integrated spectra**. Source paths, scenario IDs, units and geometry labels remain in the CSVs; they must stay attached when selecting rows.

In [ ]:
for name, frame in {"spe_hourly": spe_hourly, "shielding_dose": shielding_dose, "spe_scenarios": spe_scenarios}.items():
    print(f"{name}: {len(frame):,} rows")
    display(frame.head(3))
display(shielding_dose.groupby(["scenario_id", "dose_quantity", "dose_unit", "geometry"], dropna=False).size().rename("rows").reset_index())
print("Hourly rows with missing measurements:", int(spe_hourly.missing_measurement.sum()))


## Available charts

- 01–03: historical OMNI flux, selected shielding results, and SAPPHIRE spectra.
- [04 — Synthetic tissue-target absorbed-dose rate](charts/04_radiation_dose_rate.py): main and extreme events, external and cabin references.
- [05 — Cumulative tissue-target dose](charts/05_cumulative_exposure.py): configurable seven-hour interval, with its accumulated modeled tissue-target dose displayed.
- [06 — Shielding sensitivity](charts/06_shielding_sensitivity.py): both SAPPHIRE scenarios at all six simulated shielding levels.
- 07: separate clinical reference, not a health classification of mission results.
- 08: synthetic hourly proton fluence.

Each figure has a `.txt` caption. The older King-only figures remain under `outputs/charts/historical/`.

In [ ]:
CHART = 8
RUN_CHART = False
if RUN_CHART:
    from IPython.display import Image
    scripts = {1: "01_spe_proton_flux.py", 2: "02_shielding_vs_dose.py", 3: "03_sapphire_spectra.py",
               4: "04_radiation_dose_rate.py", 5: "05_cumulative_exposure.py", 6: "06_shielding_sensitivity.py",
               7: "07_absorbed_dose_health_reference.py", 8: "08_synthetic_sapphire_proton_profile.py"}
    completed = subprocess.run([sys.executable, str(ROOT / "charts" / scripts[CHART])], capture_output=True, text=True)
    print(completed.stdout.strip())
    if completed.returncode:
        print(completed.stderr.strip())
    else:
        display(Image(filename=completed.stdout.strip().splitlines()[-1]))

## Model-Ready Data

The verified SAPPHIRE 100-year and 1,000-year SHIELDOSE-2 event doses are distributed over the **complete selected historical event** (264 hours for August 1972; 336 for GOES-10) using normalized proton-flux timing. The default 72-hour window starts at template hour 48. August 1972 has eight missing observations, filled only in modeled copies; GOES-10 has none. nine timing alternatives retain the existing gap methods and energy-channel proxies. Shading describes sensitivity, not confidence intervals.

Both simulations use tissue at the centre of aluminium spheres, 4π irradiation, and no proton nuclear attenuation. All six depths (0.4, 1, 2, 5, 10 and 20 g/cm²) are retained. Source rad values are converted using **1 rad = 10 mGy**. These are tissue-target calculations, not astronaut whole-body doses. The spectrum is assumed fixed through time; historical timing does not set event magnitude.

The new source reports identify the return periods and match the existing spectra. The dose tables match their numerical exports at report precision; run pairing and event-integrated interpretation were confirmed by the user. Old ambiguous SHIELDOSE and failed MULASSIS outputs remain excluded. No King dose normalizes these synthetic events.

Change `radiation_config.json` for the window and interval. Select one historical template, scenario, location and timing member before integration. The same selected baseline serves all strategies; Strategy A is implemented separately.

**Quantity interpretation:** tissue-target dose is energy deposited per unit mass in an idealized target. Organ dose requires an organ-specific/anatomical model. Whole-body exposure requires a defined anatomical exposure quantity. Effective dose equivalent is in Sv and is not absorbed dose in Gy. The clinical references in Chart 07 cannot directly classify these target doses or validate EVA/cabin hardware.

In [ ]:
import json
subprocess.run([sys.executable, str(ROOT / "radiation_model.py")], check=True)
audit = json.loads((ROOT / "outputs/model/sapphire_transport_audit.json").read_text())
print("Synthetic dose status:", audit["status"])
display(pd.DataFrame(audit["scenarios"]))
for message in audit["missing_matches"]:
    print("MISSING:", message)
temporal_weights = pd.read_csv(ROOT / "outputs/model/synthetic_sapphire_temporal_weights.csv")
display(temporal_weights.groupby("member_id").normalized_bin_weight.sum().rename("full_event_weight_sum"))
mission_path = ROOT / "data/processed/mission_radiation_environment.csv"
from radiation_model import load_baseline
selected_template = audit["config"]["historical_template"]
mission_radiation_environment = load_baseline(historical_template=selected_template) if mission_path.exists() else None
if mission_radiation_environment is None:
    print("No synthetic absorbed-dose CSV: verified transport inputs are required.")
else:
    display(mission_radiation_environment.head())
if mission_radiation_environment is not None:
    central = mission_radiation_environment.query("estimate_type == 'central'")
    display(central.groupby(['scenario', 'shielding_g_cm2']).agg(
        event_total_mGy=('event_absorbed_dose_mGy', 'first'),
        full_event_total_mGy=('cumulative_end_dose_mGy', 'max')))
    interval = pd.read_csv(ROOT / "outputs/model/synthetic_sapphire_interval_exposure.csv")
    display(interval.query("estimate_type == 'central' and location in ['external_eva_reference', 'spacecraft_cabin_reference']"))

# Query storm-relative hours directly from the shared CSV.
from radiation_model import baseline_interval_dose
print("Storm hours 16–23, 0.4 g/cm², main scenario (tissue-target mGy):",
      baseline_interval_dose("SAPPHIRE-1-in-100", 0.4, 16, 23, historical_template=selected_template))
